In [4]:
from google.colab import files
uploaded = files.upload()



Saving Social_Engine_Posts_Cleaned (1).csv to Social_Engine_Posts_Cleaned (1).csv
Saving Social_Engine_Users_Cleaned (1).csv to Social_Engine_Users_Cleaned (1).csv


In [5]:
import duckdb
import pandas as pd

con = duckdb.connect("data_vortex_phase2.duckdb")

print("DuckDB connected successfully!")

DuckDB connected successfully!


In [6]:
con.execute("""
CREATE OR REPLACE TABLE posts AS
SELECT *
FROM read_csv_auto('Social_Engine_Posts_Cleaned (1).csv');
""")

con.execute("""
CREATE OR REPLACE TABLE users AS
SELECT *
FROM read_csv_auto('Social_Engine_Users_Cleaned (1).csv');
""")

print("Posts and Users tables created successfully!")

Posts and Users tables created successfully!


In [7]:
print("POSTS TABLE")
display(con.execute("DESCRIBE posts").df())

print("\nUSERS TABLE")
display(con.execute("DESCRIBE users").df())

POSTS TABLE


,column_name,column_type,null,key,default,extra
0,post_id,VARCHAR,YES,None,None,None
1,user_id,VARCHAR,YES,None,None,None
2,platform,VARCHAR,YES,None,None,None
3,text_content,VARCHAR,YES,None,None,None
4,timestamp,TIMESTAMP,YES,None,None,None
5,likes,DOUBLE,YES,None,None,None
6,shares,BIGINT,YES,None,None,None
7,comments,BIGINT,YES,None,None,None



USERS TABLE


,column_name,column_type,null,key,default,extra
0,user_id,VARCHAR,YES,None,None,None
1,location,VARCHAR,YES,None,None,None
2,language,VARCHAR,YES,None,None,None
3,account_created,DATE,YES,None,None,None
4,follower_count,BIGINT,YES,None,None,None


In [8]:
print("Posts:", con.execute("SELECT COUNT(*) FROM posts").fetchone()[0])
print("Users:", con.execute("SELECT COUNT(*) FROM users").fetchone()[0])

Posts: 12000
Users: 1500


In [9]:
con.execute("""
SELECT *
FROM posts
LIMIT 5;
""").df()

,post_id,user_id,platform,text_content,timestamp,likes,shares,comments
0,to64mgey2v3y,user_vfxs1pry,Reddit,Bummed out with my new Air Max from Nike! Abso...,2024-09-25 00:00:00,4488.0,1456,673
1,7f0wdauzbj89,user_8l7rv5oe,Reddit,My one month review of Pepsi Crystal Pepsi: Hi...,2024-08-01 16:14:00,789.0,1484,39
2,dvvhg8eel45x,user_nfo3ih5u,Unknown,Just unboxed my new Highlander from Toyota. Ex...,2025-04-13 20:12:18,2436.0,1410,839
3,hgb9cxke4t7b,user_27aje6ur,Facebook,Comparing Pepsi Crystal Pepsi to the competiti...,2024-09-10 00:00:00,167.0,584,833
4,zwerpw3wk320,user_9px5q0by,Reddit,My one week review of Coca-Cola Diet Coke: Bes...,2024-05-31 00:00:00,4749.0,152,630


In [10]:
con.execute("""
SELECT *
FROM users
LIMIT 5;
""").df()

,user_id,location,language,account_created,follower_count
0,user_guglt5jp,"Berlin, Germany",ja,2023-10-23,38963
1,user_b1p3wz81,"Munich, Germany",zh,2023-05-06,24644
2,user_jm2grmis,"Dubai, UAE",en,2023-01-03,45842
3,user_ykoqww5l,"Paris, France",fr,2023-10-11,665
4,user_3zzvv8vp,"Los Angeles, USA",ru,2023-05-23,42908


In [11]:
con.execute("""
CREATE OR REPLACE VIEW post_engagement AS
SELECT
    post_id,
    user_id,
    platform,
    timestamp,
    likes,
    shares,
    comments,
    likes + shares + comments AS total_engagement
FROM posts;
""")

print("post_engagement view created successfully!")

post_engagement view created successfully!


In [12]:
con.execute("""
SELECT *
FROM post_engagement
LIMIT 10;
""").df()

,post_id,user_id,platform,timestamp,likes,shares,comments,total_engagement
0,to64mgey2v3y,user_vfxs1pry,Reddit,2024-09-25 00:00:00,4488.0,1456,673,6617.0
1,7f0wdauzbj89,user_8l7rv5oe,Reddit,2024-08-01 16:14:00,789.0,1484,39,2312.0
2,dvvhg8eel45x,user_nfo3ih5u,Unknown,2025-04-13 20:12:18,2436.0,1410,839,4685.0
3,hgb9cxke4t7b,user_27aje6ur,Facebook,2024-09-10 00:00:00,167.0,584,833,1584.0
4,zwerpw3wk320,user_9px5q0by,Reddit,2024-05-31 00:00:00,4749.0,152,630,5531.0
5,f85c8vm8ac8l,user_aedwfomw,Unknown,2025-01-24 00:00:00,3694.0,192,135,4021.0
6,19ys654yi1if,user_27aje6ur,Facebook,2024-06-26 09:37:43,2571.0,1578,514,4663.0
7,5jywsz1bu0kp,user_hvqqyzoi,Twitter,2025-03-27 13:44:32,434.0,1683,785,2902.0
8,ol0ai4w126p0,user_634lriof,Twitter,2024-05-05 05:52:34,4798.0,491,818,6107.0
9,irjlew2pr8sw,user_8v39sv2i,Reddit,2025-02-11 00:00:00,2793.0,410,223,3426.0


E1 — Platform Popularity

In [13]:
e1 = con.execute("""
SELECT
    platform,
    COUNT(*) AS post_count
FROM posts
WHERE platform IS NOT NULL
  AND platform <> 'Unknown'
GROUP BY platform
ORDER BY post_count DESC
LIMIT 1;
""").df()

e1

,platform,post_count
0,Facebook,2074


E2 — Most Engaged Posts

In [14]:
e2 = con.execute("""
SELECT
    post_id,
    user_id,
    platform,
    likes,
    shares,
    comments,
    likes + shares + comments AS total_engagement
FROM posts
WHERE likes IS NOT NULL
ORDER BY total_engagement DESC
LIMIT 10;
""").df()

e2

,post_id,user_id,platform,likes,shares,comments,total_engagement
0,ycjj5zzt7mvx,user_d9971ba6,Instagram,4983.0,1919,991,7893.0
1,wo7py9aljg3t,user_o8le7hqf,Reddit,4864.0,1981,948,7793.0
2,gmoeib832zbs,user_pe5yckyb,Facebook,4902.0,1880,982,7764.0
3,5kvuyvf38nqx,user_z0feut2e,YouTube,4923.0,1971,861,7755.0
4,pvfl3d8hj7jd,user_csluibwk,Instagram,4989.0,1840,909,7738.0
5,tdgjjylpua20,user_8nvzxsuj,Unknown,4979.0,1932,812,7723.0
6,tne7s3o4l4wd,user_lr3fagdl,Instagram,4931.0,1903,878,7712.0
7,a1kiwl618kzy,user_aaiari8o,Facebook,4811.0,1952,920,7683.0
8,fp89q1ickn9w,user_h4lueh1i,Twitter,4740.0,1933,955,7628.0
9,5n161ir5hhhr,user_u98jwp3f,YouTube,4751.0,1981,878,7610.0


E3 — Average Engagement by Platform

In [15]:
e3 = con.execute("""
SELECT
    platform,
    ROUND(AVG(likes), 2) AS avg_likes,
    ROUND(AVG(shares), 2) AS avg_shares,
    ROUND(AVG(comments), 2) AS avg_comments,
    ROUND(AVG(likes + shares + comments), 2) AS avg_total_engagement
FROM posts
WHERE platform IS NOT NULL
  AND platform <> 'Unknown'
GROUP BY platform
ORDER BY avg_total_engagement DESC;
""").df()

e3

,platform,avg_likes,avg_shares,avg_comments,avg_total_engagement
0,YouTube,2548.00,1011.83,504.38,4064.20
1,Instagram,2496.19,1040.84,499.80,4036.83
2,Facebook,2534.89,984.17,506.94,4026.00
3,Reddit,2489.65,1002.23,511.18,4003.05
4,Twitter,2437.86,1005.39,506.13,3949.38


E4 — Highly Shared but Poorly Liked

In [16]:
e4 = con.execute("""
SELECT
    post_id,
    platform,
    likes,
    shares,
    comments
FROM posts
WHERE shares > 1500
  AND likes < 500
ORDER BY shares DESC;
""").df()

e4

,post_id,platform,likes,shares,comments
0,euvr0r10wrj6,Facebook,453.0,2000,408
1,oiszojqm6qnn,Instagram,390.0,1999,858
2,f2e5kdfldedz,Unknown,447.0,1997,641
3,mgv7p46wzpek,Reddit,252.0,1993,971
4,u1aa801qvxeu,Unknown,162.0,1992,306
...,...,...,...,...,...
236,24unyugclgt2,Twitter,121.0,1504,693
237,yynwmirnyok1,Facebook,432.0,1503,240
238,4y1wvk8hbwgd,YouTube,332.0,1503,255
239,t7svznqlhki3,Instagram,269.0,1502,51


E5 — Users With Large Audiences

In [17]:
e5 = con.execute("""
SELECT
    user_id,
    location,
    language,
    follower_count
FROM users
WHERE follower_count > 40000
ORDER BY follower_count DESC;
""").df()

e5

,user_id,location,language,follower_count
0,user_3o7w66o2,"Berlin, Germany",ja,49944
1,user_u98jwp3f,"Chicago, USA",zh,49936
2,user_usts5yuo,"Shanghai, China",ja,49933
3,user_d4eat3v3,"Tokyo, Japan",de,49914
4,user_siuvpkza,"Chicago, USA",en,49905
...,...,...,...,...
288,user_aqwezplu,"Cairo, Egypt",hi,40191
289,user_fumgsyfe,"Los Angeles, USA",es,40156
290,user_lafp17kc,"Shanghai, China",zh,40151
291,user_z05ow6gf,"Melbourne, Australia",ja,40052


M1 — Which Locations Generate the Most Engagement?

In [18]:
m1 = con.execute("""
SELECT
    u.location,
    COUNT(p.post_id) AS post_count,
    SUM(p.likes + p.shares + p.comments) AS total_engagement
FROM users u
JOIN posts p
    ON u.user_id = p.user_id
GROUP BY u.location
ORDER BY total_engagement DESC;
""").df()

m1

,location,post_count,total_engagement
0,"Los Angeles, USA",459,1837624.0
1,"Munich, Germany",452,1812278.0
2,"Shanghai, China",451,1796793.0
3,"Barcelona, Spain",439,1784336.0
4,"Melbourne, Australia",421,1715222.0
5,"Dubai, UAE",421,1706559.0
6,"Houston, USA",422,1671312.0
7,"Rio de Janeiro, Brazil",414,1636032.0
8,"Mumbai, India",413,1634881.0
9,"Osaka, Japan",398,1634303.0


M2 — Do High-Follower Users Get More Engagement?

In [19]:
m2 = con.execute("""
SELECT
    CASE
        WHEN u.follower_count >= 25000 THEN 'High Follower'
        ELSE 'Low Follower'
    END AS follower_group,
    COUNT(p.post_id) AS post_count,
    ROUND(AVG(p.likes + p.shares + p.comments), 2) AS avg_engagement_per_post
FROM users u
JOIN posts p
    ON u.user_id = p.user_id
GROUP BY follower_group
ORDER BY avg_engagement_per_post DESC;
""").df()

m2

,follower_group,post_count,avg_engagement_per_post
0,High Follower,5925,4009.48
1,Low Follower,6075,4007.91


M3 — Most Active Users

In [20]:
m3 = con.execute("""
SELECT
    u.user_id,
    u.follower_count,
    u.location,
    COUNT(p.post_id) AS post_count,
    SUM(p.likes + p.shares + p.comments) AS total_engagement
FROM users u
JOIN posts p
    ON u.user_id = p.user_id
GROUP BY
    u.user_id,
    u.follower_count,
    u.location
ORDER BY post_count DESC
LIMIT 10;
""").df()

m3

,user_id,follower_count,location,post_count,total_engagement
0,user_zqv2vrf5,13531,"Vancouver, Canada",22,90796.0
1,user_nfo3ih5u,40429,"Barcelona, Spain",19,77019.0
2,user_n0ok02rt,2531,"Dubai, UAE",18,64207.0
3,user_q3vs0uj7,32714,"Manchester, UK",18,69039.0
4,user_8wk0j5ae,36943,"Los Angeles, USA",18,62937.0
5,user_0irp4abu,44206,"Toronto, Canada",17,71943.0
6,user_hdas0iau,1620,"Rio de Janeiro, Brazil",16,64006.0
7,user_uerv85na,1824,"Rome, Italy",16,70886.0
8,user_8w06w1qu,41923,"Melbourne, Australia",16,55783.0
9,user_xetn4exw,14902,"Dubai, UAE",16,62175.0


M4 — Platform Behaviour by High-Follower Users

In [21]:
m4 = con.execute("""
SELECT
    p.platform,
    COUNT(p.post_id) AS post_count,
    ROUND(AVG(p.likes + p.shares + p.comments), 2) AS avg_engagement_per_post
FROM users u
JOIN posts p
    ON u.user_id = p.user_id
WHERE u.follower_count >= 30000
  AND p.platform <> 'Unknown'
GROUP BY p.platform
ORDER BY avg_engagement_per_post DESC;
""").df()

m4

,platform,post_count,avg_engagement_per_post
0,Instagram,809,4124.00
1,Facebook,827,4000.98
2,YouTube,846,3994.78
3,Reddit,799,3989.73
4,Twitter,785,3973.94


M5 — Detect Suspicious Engagement

In [22]:
m5 = con.execute("""
SELECT
    post_id,
    user_id,
    platform,
    likes,
    shares,
    comments,
    likes + shares + comments AS total_engagement
FROM posts
WHERE shares > likes + comments
ORDER BY shares DESC
LIMIT 20;
""").df()

m5

,post_id,user_id,platform,likes,shares,comments,total_engagement
0,euvr0r10wrj6,user_irfxbf0q,Facebook,453.0,2000,408,2861.0
1,oiszojqm6qnn,user_rscyqide,Instagram,390.0,1999,858,3247.0
2,2xcg9ld7du67,user_6w7t1ijr,Twitter,1425.0,1999,440,3864.0
3,qq86lkjrfzlt,user_wouds22l,YouTube,897.0,1999,850,3746.0
4,sjv1fkkjr8e1,user_evp5iscs,Unknown,1524.0,1998,129,3651.0
5,rryxmp0nra55,user_nf5l9rnd,Instagram,981.0,1997,958,3936.0
6,zvj4ja8bp4xx,user_zmk7o7v0,Unknown,1503.0,1997,338,3838.0
7,f2e5kdfldedz,user_kk1qdsq3,Unknown,447.0,1997,641,3085.0
8,x8wq022t0pa5,user_eatb59cy,Unknown,1380.0,1997,201,3578.0
9,lx50tyodyt6m,user_4ol58cyf,Instagram,1545.0,1996,63,3604.0


H1 — Users With Abnormally High Engagement

In [25]:
h1_check = con.execute("""
SELECT
    AVG(likes + shares + comments) AS overall_avg
FROM posts;
""").df()

h1_check

,overall_avg
0,4008.68775


In [26]:
h1 = con.execute("""
WITH user_engagement AS (
    SELECT
        u.user_id,
        u.location,
        u.follower_count,
        COUNT(p.post_id) AS post_count,
        AVG(p.likes + p.shares + p.comments) AS avg_engagement
    FROM users u
    JOIN posts p
        ON u.user_id = p.user_id
    GROUP BY
        u.user_id,
        u.location,
        u.follower_count
),
overall AS (
    SELECT AVG(likes + shares + comments) AS overall_avg_engagement
    FROM posts
)
SELECT
    ue.user_id,
    ue.location,
    ue.follower_count,
    ue.post_count,
    ROUND(ue.avg_engagement, 2) AS avg_engagement
FROM user_engagement ue
CROSS JOIN overall o
WHERE ue.avg_engagement > 2 * o.overall_avg_engagement
ORDER BY ue.avg_engagement DESC;
""").df()

print("Rows returned:", len(h1))
h1

Rows returned: 0


,user_id,location,follower_count,post_count,avg_engagement


In [27]:
con.execute("""
WITH user_avg AS (
    SELECT
        user_id,
        AVG(likes + shares + comments) AS avg_engagement
    FROM posts
    GROUP BY user_id
)
SELECT
    ROUND(MAX(avg_engagement), 2) AS highest_user_avg,
    ROUND(2 * AVG(likes + shares + comments), 2) AS required_threshold
FROM user_avg, posts;
""").df()

,highest_user_avg,required_threshold
0,5713.29,8017.38


H2 — Rank Users Within Their Location

In [28]:
h2 = con.execute("""
WITH user_totals AS (
    SELECT
        u.user_id,
        u.location,
        SUM(p.likes + p.shares + p.comments) AS total_engagement
    FROM users u
    JOIN posts p
        ON u.user_id = p.user_id
    GROUP BY
        u.user_id,
        u.location
),
ranked AS (
    SELECT
        user_id,
        location,
        total_engagement,
        ROW_NUMBER() OVER (
            PARTITION BY location
            ORDER BY total_engagement DESC
        ) AS location_rank
    FROM user_totals
)
SELECT
    user_id,
    location,
    total_engagement,
    location_rank
FROM ranked
WHERE location_rank <= 3
ORDER BY location, location_rank;
""").df()

h2

,user_id,location,total_engagement,location_rank
0,user_nfo3ih5u,"Barcelona, Spain",77019.0,1
1,user_24wzfb8b,"Barcelona, Spain",58040.0,2
2,user_5s9ifp0y,"Barcelona, Spain",57950.0,3
3,user_ttlouvlq,"Beijing, China",52359.0,1
4,user_3jedn9am,"Beijing, China",51265.0,2
...,...,...,...,...
94,user_cpxksz4r,"Toronto, Canada",56351.0,2
95,user_ukkkyp0o,"Toronto, Canada",50653.0,3
96,user_zqv2vrf5,"Vancouver, Canada",90796.0,1
97,user_i1e1kek5,"Vancouver, Canada",64836.0,2


H3 — Platform Performance Compared With Its Own Average

In [29]:
h3 = con.execute("""
WITH platform_avg AS (
    SELECT
        platform,
        AVG(likes + shares + comments) AS avg_engagement
    FROM posts
    WHERE platform IS NOT NULL
      AND platform <> 'Unknown'
    GROUP BY platform
)
SELECT
    p.post_id,
    p.platform,
    p.likes,
    p.shares,
    p.comments,
    p.likes + p.shares + p.comments AS total_engagement,
    ROUND(pa.avg_engagement, 2) AS platform_avg
FROM posts p
JOIN platform_avg pa
    ON p.platform = pa.platform
WHERE p.likes + p.shares + p.comments >= 2 * pa.avg_engagement
ORDER BY p.platform, total_engagement DESC;
""").df()

print("Rows returned:", len(h3))
h3

Rows returned: 0


,post_id,platform,likes,shares,comments,total_engagement,platform_avg


H4 — Follower-to-Engagement Anomaly

In [31]:
h4 = con.execute("""
WITH user_totals AS (
    SELECT
        u.user_id,
        u.location,
        u.follower_count,
        SUM(p.likes + p.shares + p.comments) AS total_engagement
    FROM users u
    JOIN posts p
        ON u.user_id = p.user_id
    GROUP BY
        u.user_id,
        u.location,
        u.follower_count
),
ranked AS (
    SELECT
        *,
        NTILE(10) OVER (
            ORDER BY total_engagement DESC
        ) AS engagement_decile
    FROM user_totals
)
SELECT
    user_id,
    location,
    follower_count,
    total_engagement,
    engagement_decile
FROM ranked
WHERE follower_count < 5000
  AND engagement_decile = 1
ORDER BY total_engagement DESC;
""").df()

print("Rows returned:", len(h4))
h4

Rows returned: 16


,user_id,location,follower_count,total_engagement,engagement_decile
0,user_uerv85na,"Rome, Italy",1824,70886.0,1
1,user_n0ok02rt,"Dubai, UAE",2531,64207.0,1
2,user_hdas0iau,"Rio de Janeiro, Brazil",1620,64006.0,1
3,user_fgjkkrie,"Lyon, France",2211,61500.0,1
4,user_r7eg1rac,"Houston, USA",2052,56849.0,1
5,user_rr1uzkql,"Milan, Italy",1069,55654.0,1
6,user_6wra58f7,"Johannesburg, South Africa",4953,55579.0,1
7,user_5oe5t3js,"London, UK",3151,55249.0,1
8,user_67hyf45u,"Vancouver, Canada",898,52090.0,1
9,user_zwtxksrs,"Houston, USA",1009,50627.0,1


In [32]:
uploaded = files.upload()

Saving Social_Engine_Posts_Corrupted.csv to Social_Engine_Posts_Corrupted.csv


In [33]:
import os

filename = list(uploaded.keys())[0]
print("Uploaded file:", filename)

Uploaded file: Social_Engine_Posts_Corrupted.csv


In [34]:
con.execute(f"""
CREATE OR REPLACE TABLE raw_posts AS
SELECT *
FROM read_csv_auto('{filename}');
""")

print("raw_posts loaded successfully")

raw_posts loaded successfully


In [35]:
con.execute("""
SELECT COUNT(*) AS row_count
FROM raw_posts;
""").df()

,row_count
0,12360


In [36]:
con.execute("""
DESCRIBE raw_posts;
""").df()

,column_name,column_type,null,key,default,extra
0,post_id,VARCHAR,YES,None,None,None
1,user_id,VARCHAR,YES,None,None,None
2,platform,VARCHAR,YES,None,None,None
3,text_content,VARCHAR,YES,None,None,None
4,timestamp,VARCHAR,YES,None,None,None
5,likes,VARCHAR,YES,None,None,None
6,shares,BIGINT,YES,None,None,None
7,comments,BIGINT,YES,None,None,None


In [37]:
con.execute("""
SELECT post_id, likes
FROM raw_posts
WHERE TRY_CAST(likes AS DOUBLE) < 0
LIMIT 20;
""").df()

,post_id,likes
0,mbykzpzh1l1y,-4812.0
1,wal886ggp7kg,-1795.0
2,38tm1xfxxsdi,-3707.0
3,yktgh97ttawi,-4543.0
4,loscndnlqi7b,-3194.0
5,hvjh643oxb8e,-3616.0
6,0c0tc7wbhqdu,-1165.0
7,dnhsqcx3t94w,-738.0
8,5fbt4nzjxol9,-851.0
9,whsvb3faauwf,-1205.0


In [38]:
con.execute("""
SELECT post_id, platform
FROM raw_posts
WHERE platform IS NULL
   OR TRIM(platform) = ''
LIMIT 20;
""").df()

,post_id,platform
0,dvvhg8eel45x,None
1,f85c8vm8ac8l,None
2,cpiun4hvwfns,None
3,mbykzpzh1l1y,None
4,i6io9jo0vpxa,None
5,d25e0a89eti4,None
6,sqdh2jbaeloe,None
7,ym0fk6gqthz3,None
8,df1al5tr1oca,None
9,5gyyfd2wv95z,None


In [39]:
con.execute("""
SELECT post_id, text_content
FROM raw_posts
WHERE text_content IS NULL
   OR TRIM(text_content) = ''
LIMIT 20;
""").df()

,post_id,text_content
0,k9tvoyfmg8az,None
1,0eupqhh38ego,None
2,jhzjp26xujxh,None
3,aq3dtllhs6dr,None
4,1fu318s17aoz,None
5,6ix3i5q2grjy,None
6,rcmwfuxl5rq7,None
7,62q3q1y4rvye,None
8,mqrwtcmg6k9v,None
9,o70j2gzc3kkw,None


In [40]:
con.execute("""
SELECT post_id, text_content
FROM raw_posts
WHERE regexp_matches(
    text_content,
    '&[A-Za-z][A-Za-z0-9]+;|</?[A-Za-z][^>]*>'
)
LIMIT 20;
""").df()

,post_id,text_content
0,to64mgey2v3y,Bummed out with my new Air Max from Nike! Abso...
1,ckz4ciuuw7c5,Has anyone else experienced delivery delays wi...
2,xd96337v76wx,Just tried the Superstar from Adidas. Wouldn't...
3,s5rqx7wko7m1,Confused about with my new Sienna from Toyota!...
4,2r1k9opcv2z7,Comparing Apple iMac to the competition. Highl...
5,p7h083r7vums,Just saw an ad for Samsung Neo QLED TV during ...
6,wd436n6ouxir,How do I fix about Google's Pixel Watch? @Cele...
7,gqndmf8vvpb2,NULL&amp;
8,zg7a98dozipg,Just unboxed my new Epic React from Nike. Exce...
9,9aebqwidw4n6,Has anyone else experienced software bugs with...
